# Peak Integration

This tutorial shows how to find peak maxima and determine peak areas with
SpectroChemPy. As a prerequisite,
the user is expected to have read the [Import](../importexport/import.rst),
[Import IR](../importexport/importIR.rst),
[Slicing](../processing/slicing.rst), and
[Baseline Correction](../processing/baseline.rst) tutorials.

First, let's import the SpectroChemPy API.

In [ ]:
import spectrochempy as scp

Now, import some 2D data into an NDDataset object.

In [ ]:
ds = scp.read_omnic("irdata/nh4y-activation.spg")
ds

It's a series of 55 spectra.

For the demonstration, select only the first 20 on a limited region from 1250 to
1800 cm$^{-1}$ (Do not forget to
use floating numbers for slicing).

In [ ]:
X = ds[:20, 1250.0:1800.0]

We can also eventually remove the offset on the acquisition time dimension (y).

In [ ]:
X.y -= X.y[0]
X.y.ito("min")
X.y.title = "acquisition time"

We set some plotting preferences and then plot the raw data.

In [ ]:
prefs = scp.preferences
prefs.figure.figsize = (6, 3)
prefs.colormap = "Dark2"
prefs.colorbar = True
_ = X.plot()

Now we can perform some baseline correction.

In [ ]:
blc = scp.Baseline()
blc.ranges = (
    [1740.0, 1800.0],
    [1550.0, 1570.0],
    [1250.0, 1300.0],
)  # define 3 regions where we want the baseline to reach zero.
blc.model = "polynomial"
blc.order = 3

_ = blc.fit(X)  # fit the baseline

Xcorr = blc.corrected  # get the corrected dataset
_ = Xcorr.plot()

To integrate each row on the full range, we can use the sum or trapz method of an
NDDataset.

In [ ]:
inttrapz = Xcorr.trapezoid(dim="x")
intsimps = Xcorr.simpson(dim="x")

`intsimps` is a numerical integration.  `dataset.simpson(dim="x")` and the
equivalent top-level `scp.simpson(dataset, dim="x")` produce the same result.
Do not confuse this integration function with `scp.read_simpson(...)`, which
reads SIMPSON NMR simulation data files — the two are unrelated operations
that happen to share the "simpson" name.

As you can see, both methods give almost the same results in this case.

In [ ]:
_ = scp.plot_multiple(
    method="scatter",
    ms=5,
    datasets=[inttrapz, intsimps],
    labels=["trapezoidal rule", "Simpson's rule"],
    legend="best",
)

The difference between the trapezoidal and Simpson integration methods is visualized
below. In this case, they are
extremely close.

In [ ]:
diff = (inttrapz - intsimps) * 100.0 / intsimps
diff.title = "difference"
diff.units = "percent"
_ = diff.plot(scatter=True, ms=5)